# ฝึกโมเดล YOLOv8s บน Google Colab

เลือก **Runtime → Change runtime type → GPU** ก่อนเริ่ม แล้วรันเซลล์จากบนลงล่าง

ใช้ฝึก Worker หรือ Bucket แยกกัน โดยเปลี่ยนโค้ดดาวน์โหลดชุดข้อมูลในขั้นตอนที่ 2
และพาธ `project` ในขั้นตอนที่ 5 ให้ตรงกับโมเดลที่ต้องการฝึก


## 1. ติดตั้งไลบรารี


In [ ]:
!pip install ultralytics roboflow -q


## 2. ดาวน์โหลดชุดข้อมูลจาก Roboflow

นำโค้ดดาวน์โหลดชุดข้อมูลในรูปแบบ **YOLOv8** จาก Roboflow มาวางในเซลล์ด้านล่าง
โดยใช้ชุดข้อมูล Worker หรือ Bucket ตามโมเดลที่ต้องการฝึก

โค้ดตัวอย่างเป็นชุดข้อมูล Worker ตามต้นฉบับ ให้แทน `YOUR_ROBOFLOW_API_KEY`
ด้วย API key ของคุณตอนใช้งานบน Colab


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("thanakorns-workspace-bdoqo").project("worker-w1qpd-f9ypb")
version = project.version(7)
dataset = version.download("yolov8")

print(dataset.location)


## 3. ปรับพาธใน data.yaml

รันเซลล์นี้หลังดาวน์โหลดชุดข้อมูลเสร็จ


In [ ]:
import yaml

yaml_path = f"{dataset.location}/data.yaml"

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['train'] = f"{dataset.location}/train/images"
data['val'] = f"{dataset.location}/valid/images"
data['test'] = f"{dataset.location}/test/images"

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print(data)


## 4. ตรวจขนาดวัตถุในชุดข้อมูล

ใช้โค้ดตรวจขนาดวัตถุเดิมของต้นฉบับ เซลล์นี้ข้ามได้หากต้องการฝึกทันที


In [ ]:
import os, glob
from PIL import Image

sizes = []
for label_file in glob.glob(f"{dataset.location}/train/labels/*.txt"):
    img_file = label_file.replace("labels", "images").replace(".txt", ".jpg")
    if not os.path.exists(img_file):
        img_file = img_file.replace(".jpg", ".png")
    if not os.path.exists(img_file):
        continue
    w, h = Image.open(img_file).size
    with open(label_file) as f:
        for line in f:
            parts = line.split()
            bw, bh = float(parts[3]), float(parts[4])
            sizes.append((bw * w, bh * h))

small = [s for s in sizes if s[0] < 32 or s[1] < 32]
print(f"Total boxes: {len(sizes)}")
print(f"Small (<32px): {len(small)} ({100*len(small)/len(sizes):.1f}%)")


## 5. ฝึกโมเดล YOLOv8s

คงค่าฝึกเดิมทั้งหมด เปลี่ยนเฉพาะพาธบันทึกผลเป็นพาธของ Colab

สำหรับ Worker ใช้ `project='/content/worker_detection'`
ส่วน Bucket เปลี่ยนเป็น `project='/content/bucket_detection'`


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data=yaml_path,
    epochs=300,
    patience=50,
    imgsz=960,
    batch=16,
    device=0,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    close_mosaic=15,

    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,

    save_period=10,
    project='/content/worker_detection',  # เปลี่ยนเป็น /content/bucket_detection เมื่อฝึก Bucket
    name='v1_960_s',
    exist_ok=True
)


## 6. บันทึกผลฝึกไป Google Drive

เมื่อฝึกเสร็จ ให้รันเซลล์ด้านล่างและอนุญาตเชื่อม Google Drive
ระบบจะคัดลอกโฟลเดอร์ผลฝึกทั้งหมด รวมถึง `best.pt`, `last.pt`, `results.csv` และกราฟ
ไปที่ `MyDrive/column_pouring_training` โดยแยกโฟลเดอร์ตามพาธ Worker หรือ Bucket ที่ใช้ฝึก

หากรันเซลล์นี้ซ้ำ ไฟล์ชื่อเดิมในโฟลเดอร์ปลายทางจะถูกอัปเดตด้วยผลฝึกปัจจุบัน


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

run_dir = Path(model.trainer.save_dir)
drive_dir = Path('/content/drive/MyDrive/column_pouring_training') / run_dir.parent.name / run_dir.name

shutil.copytree(run_dir, drive_dir, dirs_exist_ok=True)
print('บันทึกผลฝึกทั้งหมดไป Google Drive แล้ว:', drive_dir)
